# agent-eval-lab — Colab walkthrough

Compare three ways of answering customer questions with an LLM, on the same 28 test cases:

| mode | what the model gets |
|---|---|
| `llm_only` | only the question |
| `rag` | the question + the 4 most relevant document passages |
| `agent` | the question + tools it can call itself (search, specs, calculator, charging cost, booking) |

**Runtime → Change runtime type → T4 GPU** (free). Open models run locally through Ollama; no API key needed.

## 1 · Setup

In [ ]:
REPO = "https://github.com/sandraesudhakaran-creator/agent-eval-lab.git"
!git clone -q $REPO /content/agent-eval-lab
%cd /content/agent-eval-lab
!pip install -q -e .
import sys; sys.path.insert(0, "/content/agent-eval-lab/src")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/agent_eval_lab"   # results survive a disconnect
!mkdir -p $OUT

## 2 · Baseline without any LLM (30 seconds)

The extractive baseline just returns the best-matching document passage. It sets the floor every LLM setup has to beat.

In [ ]:
!agentlab eval --provider extractive --out $OUT/baseline --title "Extractive baseline (no LLM)"

## 3 · Start Ollama and download open models

Both models support native tool calling. Downloading takes a few minutes.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
!ollama pull llama3.1:8b

## 4 · Try one question and look at the agent's steps

In [ ]:
!agentlab ask "What is the total price of a Ridge together with the HomeCharge wallbox?" --provider ollama --model qwen2.5:7b --trace

In [ ]:
!agentlab ask "Please book me a test drive of the Ridge in Hamburg on 2026-11-20." --provider ollama --model qwen2.5:7b --trace

## 5 · Full evaluation: 2 models × 3 modes × 28 cases

Roughly 15–30 minutes on a T4. Each model × mode prints its pass count when done.

In [ ]:
!agentlab eval --provider ollama --model qwen2.5:7b --model llama3.1:8b --out $OUT/open_models --title "Open models on Ollama (T4)"

## 6 · Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{OUT}/open_models/report.md").read()))
display(Image(f"{OUT}/open_models/pass_rate_by_category.png"))

In [ ]:
# Inspect one failure in detail: every model call and tool call with its output
import json
runs = [json.loads(l) for l in open(f"{OUT}/open_models/runs.jsonl")]
failed = [r for r in runs if not r["score"]["passed"] and r["mode"] == "agent"]
r = failed[0] if failed else runs[0]
print(r["model"], r["mode"], r["case_id"], "\nQ:", r["question"], "\nA:", r["answer"])
for s in r["steps"]:
    print(" ", s["kind"], s["name"], s["arguments"] or "", "->", (s["output"] or "")[:200])

## 7 · Optional: a larger hosted model (Groq free tier)

Create a free API key at console.groq.com, then run the same evaluation. Check the Groq console for current model names.

In [ ]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
!agentlab eval --provider groq --model llama-3.3-70b-versatile --out $OUT/groq --title "Hosted model (Groq)"

## 8 · Numbers for the README

Copy from the reports above into the README's *Results* section: pass rate per model × mode, tool recall/precision, refusal accuracy, mean latency and tokens, and one or two failure examples with your explanation. Also copy `pass_rate_by_category.png` into `docs/img/`.